# Prelude

In [1]:
%cd /home/mfixman/TP/MyPyCop

/home/mfixman/Dropbox/continuous/TP/MyPyCop


In [2]:
import pandas
import scipy
import numpy
import matplotlib
import seaborn
import json
import glob

from matplotlib import pyplot

from pandas import Series, DataFrame

In [3]:
seaborn.set()

# Getting Data

In [4]:
data = pandas.concat(
    [
        pandas.read_csv(
            'results/SATReset_Tuesday_partial.csv',
        ),
        pandas.read_csv(
            'results/SATResetTuesdayControl.csv',
        )
    ],
    ignore_index = True,
).drop_duplicates(['problem', 'strategy'], keep = 'last')

data = data[data.problem.isin(data[data.strategy == 'SATResetAtStaleBacktrack'].problem)]

FileNotFoundError: [Errno 2] No such file or directory: 'results/SATReset_Tuesday_partial.csv'

In [ ]:
problems = data.problem.str[:3].value_counts().sort_index()
pandas.concat(
    [
        problems[0::3].reset_index(),
        problems[1::3].reset_index(),
        problems[2::3].reset_index(),
    ],
    axis = 1,
).fillna('')

# Analysing

In [ ]:
control = ['FirstAction', 'LeanCopCut']

In [ ]:
data.groupby('strategy').problem.count()

In [ ]:
pandas.crosstab(
    data.strategy,
    data.result,
    margins = True,
).sort_values('All')

In [ ]:
data[data.problem == 'PUZ053-1.p']

In [ ]:
pandas.crosstab(
    data.result.fillna('?'),
    data.problem_declared_status,
)

In [ ]:
data[data.solved]

In [ ]:
data[data.problem_spc_tags.str.contains('FOF')].problem.str[6].value_counts()

In [ ]:
data[data.solved].groupby('strategy').size()

# Results

In [ ]:
data[data.solved].strategy.value_counts(ascending = True).plot.bar()

In [ ]:
data[data.strategy == 'SATResetAtStaleAction'].result.value_counts()

In [ ]:
data[data.strategy == 'SATResetAtTwo'].result.value_counts()

In [ ]:
merged = pandas.merge(
    data,
    data,
    left_on = 'problem',
    right_on = 'problem',
    suffixes = ('_left', '_right'),
)

In [ ]:
pandas.crosstab(merged.strategy_left, merged.strategy_right)

In [ ]:
values = (
    merged
    [~merged.solved_left & merged.solved_right]
    [['strategy_left', 'strategy_right']]
    .value_counts()
)

order = values.groupby(level = 'strategy_right').sum().sort_values().index

results = (
    values
    .unstack(level = 'strategy_right', fill_value = 0)
    .rename_axis(index = 'Loser', columns = 'Winner')
    .reindex(index = order, columns = order, fill_value = 0)
)

renamer = {
    'SATSolve': 'SATCoP (orig)',
    'SATSolveLiteral': 'SATCoP',
    'SATResetAtStaleAction': 'StaleAction',
	'SATResetAtStaleBacktrack': 'StaleBacktrack',
	'SATResetBandit': 'Bandit',
    'SATResetAtTwoFixedStep': 'ResetAtTwo\'',
    'SATResetAtTwo': 'ResetAtTwo',
}

cols = results.columns[results.columns.str.startswith('SAT') & ~results.columns.str.endswith('Literal')]
cols = results.columns
plot = (
    results.
    loc[cols, cols].
    rename_axis(index = 'Loser', columns = 'Winner').
    rename(index = renamer).
    rename(columns = renamer)
)

seaborn.heatmap(
    plot,
    annot = plot.astype(str).where(~numpy.eye(results.index.size, dtype = bool), other = ''),
    fmt = 's',
    cmap = 'YlOrRd',
)
# pyplot.title(f'Winners and losers of different SAT policies. Total of {merged.problem.nunique():,d} problems.')
# pyplot.savefig('figures/sat_policies_plot_29Apr.png', bbox_inches = 'tight', dpi = 400)

In [ ]:
pyplot.rcParams["figure.figsize"] = (5,6)

times = [0, 1, 5, 10, 30, 60, 90, 120]
solved_time = Series(
    numpy.select(
        [data.solved & (data.time_ms <= 1000 * x) for x in times],
        [x for x in times],
        default = None,
    ),
    index = data.index,
    name = 'solved_time',
)

q = pandas.concat([data, solved_time], axis = 1)[['strategy', 'solved_time']]
b = q.groupby('strategy').value_counts(sort = False).sort_index(level = [0, 1]).unstack().reindex(order).rename(index = renamer)
b.plot.bar(stacked = True)
pyplot.ylabel('Problems Solved')
pyplot.xlabel('Strategy')
pyplot.legend(loc = 0, fontsize = 'small', reverse = True, title = 'Time (s) ≤', title_fontsize = 'small')

pyplot.title(f'Out of {data.problem.nunique()} problems.')
None

In [ ]:
pyplot.rcParams["figure.figsize"] = (5,6)

times = [0, 32, 1000, 5000, 32000, 175000, 1000000]
solved_time = Series(
    numpy.select(
        [data.solved & (data.steps <= x) for x in times],
        [x for x in times],
        default = None,
    ),
    index = data.index,
    name = 'solved_time',
)

q = pandas.concat([data, solved_time], axis = 1)[['strategy', 'solved_time']]
b = q.groupby('strategy').value_counts(sort = False).sort_index(level = [0, 1]).unstack().reindex(order).rename(index = renamer)
b.plot.bar(stacked = True)
pyplot.ylabel('Problems Solved')
pyplot.xlabel('Strategy')
pyplot.legend(loc = 0, fontsize = 'small', reverse = True, title = 'Steps ≤', title_fontsize = 'small')